In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

combine = pd.read_csv("data/combine.csv")
draft = pd.read_csv("data/draft.csv")

combine_draft = pd.merge(left = combine, right = draft, how = "inner", on = "playerId")

# Are NFL Scouting Combine Metrics Meaningful Predictors of Quarterback Performance?
### DATASCI 151 Final Project Spring 2026
**Group Members:** Nicholas Huang [2627522],

---

## Introduction

The NFL relies on the NFL Combine, a week-long event where top collegiate football players perform a series of athletic and cognitive tests in front of teams scouts, as one of the primary ways to discern player skill and draft value. For quarterbacks, one of the most valuable positions in the game, the combine is a chance to showcase the necessary physical and processing skills that may ensure them a spot in the league. These tests include the 40-yard dash, measuring straight-line speed, vertical jump and 3-cone drill, measuring agility and explosiveness, the 20-yard shuttle, measuring short-bursts of speeds, and the Wonderlic Personnel Test, measuring coginitive ability. Teams scouts watch prospective players complete these tests which then informs their decisions on if and when they should select them in the NFL Draft. The central question motivating this project is: **do combine measurements actually predict how well a quarterback performs in real NFL games?**

Using a combination of datasets from the **NFL Play Statistics dataset** from Kaggle, an open-source data sharing platform, we first calculate three career performance statistics for each quarterback: completion percentage, touchdown-to-interception ratio, and passing yards per game. We then merge these statistics with each player's combine testing results and draft statistics to examine the correlations between combine metrics and on-field performance. Our results indicate that [FILL IN]

## Data Description

In [3]:
import pandas as pd #Importing Libraries
import numpy as np
import matplotlib.pyplot as plt

df_passer = pd.read_csv('https://raw.githubusercontent.com/tuckersampson/final_project_151/refs/heads/main/data/passer.csv') #Importing Data
df_combine = pd.read_csv('https://raw.githubusercontent.com/tuckersampson/final_project_151/refs/heads/main/data/combine.csv')
df_draft = pd.read_csv('https://raw.githubusercontent.com/tuckersampson/final_project_151/refs/heads/main/data/draft.csv')
df_gp = pd.read_csv('https://raw.githubusercontent.com/tuckersampson/final_project_151/refs/heads/main/data/gameParticipation.csv')

print("Passer") #Generating stats for dataframe length and unique observations
print("Number of rows:", len(df_passer))
print("Unique playerIds:", df_passer["playerId"].nunique())
print("Unique playIds:", df_passer["playId"].nunique())

print("\nCombine")
print("Number of rows:", len(df_combine))
print("Unique playerIds:", df_combine["playerId"].nunique())
print("Unique positions:", df_combine["combinePosition"].nunique())

print("\nDraft")
print("Number of rows:", len(df_draft))
print("Unique playerIds:", df_draft["playerId"].nunique())
print("Unique draft years:", df_draft["draft"].nunique())
print("Unique rounds:", df_draft["round"].nunique())

print("\nGameParticipation")
print("Number of rows:", len(df_gp))
print("Unique playerIds:", df_gp["playerId"].nunique())
print("Unique gameIds:", df_gp["gameId"].nunique())

Passer
Number of rows: 397265
Unique playerIds: 755
Unique playIds: 397265

Combine
Number of rows: 10080
Unique playerIds: 10078
Unique positions: 26

Draft
Number of rows: 12140
Unique playerIds: 12119
Unique draft years: 43
Unique rounds: 12

GameParticipation
Number of rows: 423185
Unique playerIds: 5827
Unique gameIds: 995


From the twenty available tables in the entire datset, we selected four for our analysis. The first, **`df_passer`** contains one row per pass attempt for every quarterback across the 2004–2019 NFL seasons, containing around 397,000 individual attempts, recording the outcome of each throw.  **`df_combine`** contains the combine testing results for all players, 10,080 unique players, who have participated in the event since the late 1980s. **`df_draft`** records the draft year, round, and pick number for all drafted players, consisting of 12,140 players. **`df_gp`** logs each player's appearance in each game, which we use to count the number of games each quarterback played in order to compute yards per game.

### Merging Data

To merge our data, we first aggregated the `df_passer` play-by-play table into one row per player with career totals. We also join `gameParticipation` to obtain each QB's total games played utilizing inner join. Second, we merge the `combine` and `draft` tables on `playerId`, also on inner join, and keep only quarterbacks. Finally, we join the combine+draft table with the passer statistics, again on `playerId`, to give us a single table with one row per QB that we can conduct analysis on.

In [ ]:
df_passer['pass_yards'] = df_passer['passLength'] * df_passer['passComp'] #Isolating only completed passes

qb_stats = df_passer.groupby('playerId').agg(attempts = ('passAtt', 'sum'), #Aggregating QB stats for each unique player
                                        completions = ('passComp', 'sum'),
                                        touchdowns = ('passTd', 'sum'),
                                        interceptions = ('passInt', 'sum'),
                                        total_yards = ('pass_yards','sum')).reset_index()

qb_games = (df_gp[df_gp['position'] == 'QB'].groupby(by = 'playerId')['gameId'].nunique()) #Counting games played per unique player

qb_stats = qb_stats.merge(qb_games, on='playerId', how='inner') #Merging games played and QB stats on inner, to preserve only players in QB stats

In [ ]:
INSERT CODE FOR MERGING DRAFT + COMBINE DATA

### Cleaning Data

Our data required minimal cleaning. For `qb_stats`, we first examined the shape, data types, and unique values of each column. We used an inner join when merging the aggregated passer data with game participation counts, which excluded the 538 quarterbacks who appeared in `passer` but had no matching record in `gameParticipation`. We then used `query()` to filter out quarterbacks with fewer than 100 career pass attempts, as their career averages would not be representative of sustained NFL performance. [INSERT STEPS FOR CLEANING DRAFT + COMBINE DATA]

In [ ]:
n_rows, n_cols = qb_stats.shape #Computing stats for dataframe
print("# of rows:", n_rows)
print("# of columns:", n_cols)
print()
print(qb_stats.dtypes)
print()

print("Sample of games_played values:", pd.unique(qb_stats["gameId"])[:10])

qb_stats = qb_stats.query("attempts >= 100") #Only keeping players with over 100 passes

qb_stats['comp_pct'] = (qb_stats['completions'] / qb_stats['attempts'] * 100).round(2) #Computing QB stats of interest
qb_stats['td_int_ratio'] = (qb_stats['touchdowns'] / (qb_stats['interceptions'] + 1)).round(2)
qb_stats["yards_per_game"] = (qb_stats["total_yards"]  / qb_stats["gameId"]).round(2)

print("# of rows after cleaning:", len(qb_stats))
print()
print("Nulls remaining:")
print(qb_stats.isnull().sum())

In [ ]:
INSERT CODE FOR CLEANING DRAFT + COMBINE DATA

### Descriptive Statistics

The table below summarizes the three computed performance metrics and the five combine measurements we will use in our analysis. Combine metrics are shown for all QBs who have a recorded value.

In [ ]:
INSERT CODE FOR DESCRIPTIVE STATISTICS

---
## Results

### Correlation Between Combine Metrics and QB Performance

In [ ]:
INSERT CODE FOR GENERATING CORRELATION AND SUMMARY TABLES

**Interpretation:** 

INSERT INTERPRETATION OF DATA

---
## Discussion

INSERT DISCUSSION OF DATA